In [ ]:
import sys
import os
import time
import yaml
sys.path.append(os.path.abspath('..')) 
from drone_env.drone_sim import RoomDroneEnv
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
import pybullet as p

# ==========================================
# CONFIGURE WHICH STAGE TO WATCH:
STAGE_TO_WATCH = 0
# ==========================================

print(f"Presentation Mode Active: Loading Stage {STAGE_TO_WATCH}...")

config_path = os.path.abspath(os.path.join('..', 'configs', 'teacher_ppo.yaml'))
with open(config_path, 'r') as file:
    config = yaml.safe_load(file)

stage_key = f"stage_{STAGE_TO_WATCH}"
stage_config = config['stages'][stage_key]
reward_weights = config['rewards']

NUM_OBS = stage_config['num_obstacles']
RAND_OBS = stage_config['randomize_obstacles']
RAND_COINS = stage_config['randomize_coins']
RUN_NAME = stage_config['run_name']

# Env oluştur ve VecNormalize ile sar
env = RoomDroneEnv(gui=True, num_obstacles=NUM_OBS, randomize_obstacles=RAND_OBS, randomize_coins=RAND_COINS,reward_weights=reward_weights)
env = DummyVecEnv([lambda: env])

model_path = os.path.abspath(os.path.join('..', 'models', RUN_NAME, 'best_model'))
vecnorm_path = f"{model_path}_vecnormalize.pkl"
print(f"Loading Brain from: {model_path}.zip")

try:
    # Ajanın kör olmaması için eğitimdeki normalizasyon istatistiklerini yüklüyoruz
    env = VecNormalize.load(vecnorm_path, env)
    env.training = False # İzlerken istatistikleri bozmaması için
    env.norm_reward = False
    
    model = PPO.load(model_path, env=env)
except Exception as e:
    print(f"ERROR: Model or Normalization stats not found!\nDetails: {e}")
    sys.exit()

# VecEnv sadece obs döner
obs = env.reset()
p.resetDebugVisualizerCamera(cameraDistance=6.0, cameraYaw=45, cameraPitch=-30, cameraTargetPosition=[0, 0, 1])

print(f"[{RUN_NAME}] Simulation started. Runs until collision...")

while True:
    action, _states = model.predict(obs, deterministic=True)
    # VecEnv 4 değer döner
    obs, rewards, dones, infos = env.step(action) 
    time.sleep(1./240.) 
    
    if dones[0]:
        print("Episode finished. Loading new room...")
        time.sleep(1) 
        # VecEnv otomatik reset atar ama piksellerin/ortamın tazelenmesi için manuel tetikliyoruz
        obs = env.reset()